# 11 — Embedding Forward and Backward

**Ladder:** one-hot / Python scatter (Method 01) → Triton gather + atomic scatter-add (Method 02) → fused lookup + position add with dtype dispatch (Method 03).

**Study tip.** An embedding is just a **gather**

$$
X = W[I],
$$

and its backward is a **scatter-add**

$$
dW[I_m] \mathrel{+}= dG_m .
$$

Write those two index maps by hand for the tiny $V=5,\ D=3,\ M=4$ example before you read the kernels. The backward pass is the expensive one because it touches the whole $V\times D$ table, even when only a few rows are used.

Source notes: [`doc/Embedding_kernal.pdf`](doc/Embedding_kernal.pdf) · Figures: [`docs/images/`](docs/images/) · Visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

**Learning objectives**

- Read an embedding lookup as a gather and as the one-hot matrix product $X=PW$.
- Derive the backward pass $dW=P^\top G$ and recognize it as scatter-add.
- Write a 2-D tiled Triton gather kernel and an atomic scatter-add kernel.
- Explain why the backward needs `tl.atomic_add`, and why bf16 gradients are accumulated in fp32.
- Fuse the lookup with a position embedding: $H=X+\mathcal P$.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl


def _active_device():
    """Pick Triton's active device, else CUDA, else CPU (study helper)."""
    try:
        return triton.runtime.driver.active.get_active_torch_device()
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    """Skip a GPU-only test when no accelerator is present."""
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


def get_dtype_flag(dtype: torch.dtype) -> int:
    """Encode the element type for the kernel: 0=fp32, 1=fp16, 2=bf16."""
    if dtype == torch.float32:
        return 0
    if dtype == torch.float16:
        return 1
    if dtype == torch.bfloat16:
        return 2
    raise TypeError(f"unsupported dtype: {dtype}")


print("device:", DEVICE, "| dtype flag fp32:", get_dtype_flag(torch.float32))


## Part I — Mathematics

### 1. Table, token ids, lookup

Let the table be

$$
W\in\mathbb R^{V\times D},
$$

with vocabulary size $V$ and embedding width $D$. A token id is an integer
$I_m\in\{0,\ldots,V-1\}$. The forward pass copies row $I_m$:

$$
X_{m,:}=W_{I_m,:},
\qquad
X=W[I].
$$

The same operation can be written as a matrix product with a **one-hot** matrix

$$
P\in\{0,1\}^{M\times V},
\qquad
P_{m,j}=\mathbf 1[j=I_m],
\qquad
X=PW.
$$

If `ids` has shape `(..., )`, the output has shape `(..., D)`: the trailing
dimension is always the embedding width.

### 2. Position embedding

A learned or fixed position table $\mathcal P\in\mathbb R^{M\times D}$ is added
elementwise:

$$
H=X+\mathcal P.
$$

Because the sum is linear,

$$
\frac{\partial L}{\partial X}=\frac{\partial L}{\partial H},
\qquad
\frac{\partial L}{\partial \mathcal P}=\frac{\partial L}{\partial H}.
$$

If $\mathcal P$ is shared across a batch axis, sum $\partial L/\partial H$ over
that axis. The fused kernel below assumes the simple same-shape case.

### 3. Backward pass

Let $G=\partial L/\partial X\in\mathbb R^{M\times D}$. From $X=PW$,

$$
dW=P^\top G,
\qquad
dW_{j,:}=\sum_{m:\,I_m=j} G_{m,:}.
$$

Every token $m$ adds its gradient row $G_m$ into row $I_m$ of $dW$. Ids are
integers, so they have no derivative; `dX` is supplied by the downstream layer.

### 4. Why the backward needs atomics

Two programs may own two different tokens $m$ and $m'$ with the same id
$I_m=I_{m'}$. A plain `tl.store` would race: the last writer wins and one
gradient row is lost. `tl.atomic_add` makes the update read-modify-write, so
both contributions survive. The sum is correct, but the **order** of the
floating-point additions is not fixed, so repeated runs can differ in the last
bits. If you need bitwise reproducibility, use a deterministic segmented
reduction instead of atomics (an exercise).

### 5. Cost model

For one forward + backward with $M$ tokens, width $D$, vocabulary $V$, and
$b$ bytes per element:

| Phase | Traffic | Dominant term |
|---|---|---|
| Forward | read $M D b$ (table rows) + write $M D b$ (output) | $2MDb$ |
| Backward | act on the full table $V D b$ + read $M D b$ (grad) | $VDb$ |

The backward is usually much larger because $V\gg M$ in language models. For
$V=50{,}000$, $D=1024$, $M=2048$, and fp16 ($b=2$):

- forward output: $MDb = 4.19$ MB;
- backward table: $VDb = 102.4$ MB.

That is the motivation for sparse or row-cached embedding gradients in
production.


### Worked numerical example (from the notes)

Use the tiny case $V=5$, $D=3$, $M=4$, with

$$
W=\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\\10&11&12\\13&14&15\end{bmatrix},
\qquad
I=\begin{bmatrix}2\\0\\1\\2\end{bmatrix}.
$$

The lookup is

$$
X=W[I]=\begin{bmatrix}7&8&9\\1&2&3\\4&5&6\\7&8&9\end{bmatrix}.
$$

Take the upstream gradient

$$
G=\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\\10&11&12\end{bmatrix}.
$$

Then

$$
dW_0=G_1=\begin{bmatrix}4&5&6\end{bmatrix},\quad
dW_1=G_2=\begin{bmatrix}7&8&9\end{bmatrix},\quad
dW_2=G_0+G_3=\begin{bmatrix}11&13&15\end{bmatrix},
$$

and rows $3,4$ stay zero. The next cell verifies exactly this.


In [ ]:
# Tiny case from the notes: V=5, D=3, M=4.
table = torch.arange(1, 16, dtype=torch.float32).view(5, 3)
ids = torch.tensor([2, 0, 1, 2])
grad = torch.arange(1, 13, dtype=torch.float32).view(4, 3)

one_hot = torch.zeros(4, 5)
one_hot[torch.arange(4), ids] = 1.0

lookup = table[ids]
matmul_form = one_hot @ table
print("P =\n", one_hot)
print("X = W[I] =\n", lookup)
print("P @ W   =\n", matmul_form)
print("same:", torch.equal(lookup, matmul_form))

# Backward: scatter-add the gradient rows into dW.
dW = torch.zeros_like(table)
dW.index_add_(0, ids, grad)
print("dW =\n", dW)
print("expected dW[2] = G[0] + G[3] =", grad[0] + grad[3])


## Part II — Method 01: naive (one-hot and Python scatter)

The notes start from the **one-hot matrix** $P$: the forward is the dense
product $X=PW$, and the backward is $dW=P^\top G$. For teaching, a Python loop
over tokens is clearer than materializing $P$; both compute the same thing.

**Forward.** For each token $m$, copy row $I_m$.

**Backward.** For each token $m$, add $G_m$ into row $I_m$.

These two loops are the reference against which both Triton kernels are
checked. On CPU they run directly; on GPU they are slow (one Python iteration
per token), which is exactly why the kernel versions exist.


In [ ]:
def naive_embedding_pseudo(table: torch.Tensor, ids: torch.Tensor) -> torch.Tensor:
    """Method 01 forward: one Python iteration per token (the gather definition)."""
    table = torch.as_tensor(table)
    ids = torch.as_tensor(ids)
    out = torch.empty(*ids.shape, table.shape[1], dtype=table.dtype, device=table.device)
    flat_ids = ids.reshape(-1)
    flat_out = out.reshape(-1, table.shape[1])
    for m in range(flat_ids.numel()):
        flat_out[m] = table[int(flat_ids[m])]
    return out


def one_hot_embedding_pseudo(table: torch.Tensor, ids: torch.Tensor) -> torch.Tensor:
    """Method 01 as a matrix product: X = P W with P[m, ids[m]] = 1."""
    table = torch.as_tensor(table)
    ids = torch.as_tensor(ids)
    flat_ids = ids.reshape(-1)
    P = torch.zeros(flat_ids.numel(), table.shape[0], dtype=table.dtype, device=table.device)
    P[torch.arange(flat_ids.numel()), flat_ids] = 1.0
    return (P @ table).reshape(*ids.shape, table.shape[1])


def naive_embedding_backward_pseudo(
    grad_out: torch.Tensor, vocab_size: int, ids: torch.Tensor
) -> torch.Tensor:
    """Method 01 backward: dW[j] += G[m] for every token m with ids[m] == j."""
    grad_out = torch.as_tensor(grad_out)
    ids = torch.as_tensor(ids)
    dW = torch.zeros(
        vocab_size, grad_out.shape[-1], dtype=grad_out.dtype, device=grad_out.device
    )
    grad_rows = grad_out.reshape(-1, grad_out.shape[-1])
    flat_ids = ids.reshape(-1)
    for m in range(flat_ids.numel()):
        dW[int(flat_ids[m])] += grad_rows[m]
    return dW


In [ ]:
def test_pseudo_matches_torch() -> None:
    """CPU-runnable: one-hot, Python gather, and Python scatter match PyTorch."""
    torch.manual_seed(0)
    table = torch.randn(11, 5)
    ids = torch.tensor([[3, 1, 4], [1, 5, 9]])
    ref = table[ids]

    assert torch.allclose(naive_embedding_pseudo(table, ids), ref)
    assert torch.allclose(one_hot_embedding_pseudo(table, ids), ref)

    grad = torch.randn_like(ref)
    ref_dw = torch.zeros_like(table)
    ref_dw.index_add_(0, ids.reshape(-1), grad.reshape(-1, 5))
    assert torch.allclose(
        naive_embedding_backward_pseudo(grad, table.shape[0], ids), ref_dw
    )
    print("method 01: forward and backward OK")


def test_worked_example() -> None:
    """Check the exact numbers printed in the notes."""
    table = torch.arange(1, 16, dtype=torch.float32).view(5, 3)
    ids = torch.tensor([2, 0, 1, 2])
    grad = torch.arange(1, 13, dtype=torch.float32).view(4, 3)

    expected_x = torch.tensor([[7, 8, 9], [1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float32)
    assert torch.equal(table[ids], expected_x)

    expected_dw = torch.zeros(5, 3)
    expected_dw[0] = torch.tensor([4.0, 5.0, 6.0])
    expected_dw[1] = torch.tensor([7.0, 8.0, 9.0])
    expected_dw[2] = torch.tensor([11.0, 13.0, 15.0])
    assert torch.equal(naive_embedding_backward_pseudo(grad, 5, ids), expected_dw)
    print("worked example: X and dW match the notes")


test_pseudo_matches_torch()
test_worked_example()


## Part III — Method 02: kernel-based (tiled gather / atomic scatter-add)

Both kernels use a **2-D grid**:

$$
\texttt{pid}_m=\texttt{tl.program\_id}(0),\qquad
\texttt{pid}_n=\texttt{tl.program\_id}(1).
$$

Program $(\texttt{pid}_m,\texttt{pid}_n)$ owns the tile

$$
\text{rows}=\texttt{pid}_m B_M+\{0,\ldots,B_M-1\},
\qquad
\text{cols}=\texttt{pid}_n B_N+\{0,\ldots,B_N-1\},
$$

with masks $\text{rows}<M$ and $\text{cols}<D$. The gather offset is

$$
\text{address}_{m,d}=I_m\cdot D+d.
$$

**Tile map for $M=4$, $D=4$, $B_M=B_N=2$:** the grid is $2\times2=4$
programs.

| pid `(pid_m,pid_n)` | rows | cols |
|---|---|---|
| `(0,0)` | 0,1 | 0,1 |
| `(0,1)` | 0,1 | 2,3 |
| `(1,0)` | 2,3 | 0,1 |
| `(1,1)` | 2,3 | 2,3 |

**Masking detail.** The id is loaded with the row mask and `other=0`, so
out-of-range rows read row 0; the load and store are still masked by
`row_mask[:, None] & col_mask[None, :]`, so those lanes never touch memory.

**Pointer-type dispatch.** The notes carry a `DTYPE_FLAG` (`0` fp32, `1` fp16,
`2` bf16) and cast the pointers inside the kernel. The wrapper computes the
flag once; the `if DTYPE_FLAG == ...` branches are resolved at compile time
because the flag is a `tl.constexpr`.


In [ ]:
@triton.jit
def embedding_forward_kernel(
    table_ptr,          # W: [V, D]
    ids_ptr,            # I: [M] int32
    out_ptr,            # X: [M, D]
    n_tokens,           # M (runtime)
    embed_dim,          # D (runtime)
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,   # 0=fp32, 1=fp16, 2=bf16
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # The notes dispatch on DTYPE_FLAG so one kernel serves all three dtypes.
    if DTYPE_FLAG == 0:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.float32))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.float32))
    elif DTYPE_FLAG == 1:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.float16))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.float16))
    else:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.bfloat16))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.bfloat16))
    ids_ptr = tl.cast(ids_ptr, tl.pointer_type(tl.int32))

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < n_tokens
    col_mask = cols < embed_dim
    mask = row_mask[:, None] & col_mask[None, :]

    # Load the token ids for this row tile; masked lanes use id 0.
    ids = tl.load(ids_ptr + rows, mask=row_mask, other=0)

    # Gather W[ids[m], cols] for every (m, col) in the tile.
    table_offsets = ids[:, None].to(tl.int64) * embed_dim + cols[None, :]
    values = tl.load(table_ptr + table_offsets, mask=mask, other=0.0)

    # X[m, col] = the gathered value.
    out_offsets = rows[:, None].to(tl.int64) * embed_dim + cols[None, :]
    tl.store(out_ptr + out_offsets, values, mask=mask)


In [ ]:
def embedding_forward(
    table: torch.Tensor,
    ids: torch.Tensor,
    BLOCK_M: int = 128,
    BLOCK_N: int = 128,
    check_indices: bool = False,
) -> torch.Tensor:
    """Method 02 forward: X = W[I] with a 2-D grid over (tokens, embedding dim)."""
    assert table.ndim == 2, "table must be [V, D]"
    table = table.contiguous()
    ids = ids.contiguous()

    if check_indices and ids.numel() > 0:
        # A bounds check forces a GPU sync, so it is opt-in (as in the notes).
        assert ids.min() >= 0, "negative token id"
        assert ids.max() < table.shape[0], "token id out of range"

    ids_i32 = ids.to(torch.int32)
    out = torch.empty(*ids.shape, table.shape[1], device=table.device, dtype=table.dtype)
    n_tokens = ids.numel()
    if n_tokens == 0:
        return out

    grid = (
        triton.cdiv(n_tokens, BLOCK_M),
        triton.cdiv(table.shape[1], BLOCK_N),
    )
    embedding_forward_kernel[grid](
        table,
        ids_i32,
        out,
        n_tokens,
        table.shape[1],
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        DTYPE_FLAG=get_dtype_flag(table.dtype),
    )
    return out


In [ ]:
@triton.jit
def embedding_backward_kernel(
    grad_out_ptr,       # G: [M, D]
    grad_weight_ptr,    # dW: [V, D]
    ids_ptr,            # I: [M] int32
    n_tokens,           # M
    embed_dim,          # D
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    if DTYPE_FLAG == 0:
        grad_out_ptr = tl.cast(grad_out_ptr, tl.pointer_type(tl.float32))
        grad_weight_ptr = tl.cast(grad_weight_ptr, tl.pointer_type(tl.float32))
    elif DTYPE_FLAG == 1:
        grad_out_ptr = tl.cast(grad_out_ptr, tl.pointer_type(tl.float16))
        grad_weight_ptr = tl.cast(grad_weight_ptr, tl.pointer_type(tl.float16))
    else:
        grad_out_ptr = tl.cast(grad_out_ptr, tl.pointer_type(tl.bfloat16))
        grad_weight_ptr = tl.cast(grad_weight_ptr, tl.pointer_type(tl.bfloat16))
    ids_ptr = tl.cast(ids_ptr, tl.pointer_type(tl.int32))

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < n_tokens
    col_mask = cols < embed_dim
    mask = row_mask[:, None] & col_mask[None, :]

    ids = tl.load(ids_ptr + rows, mask=row_mask, other=0)

    # Read the gradient rows owned by this tile.
    grad_offsets = rows[:, None].to(tl.int64) * embed_dim + cols[None, :]
    grad = tl.load(grad_out_ptr + grad_offsets, mask=mask, other=0.0)

    # dW[ids[m], cols] += G[m, cols].  Atomic because ids can repeat.
    weight_offsets = ids[:, None].to(tl.int64) * embed_dim + cols[None, :]
    tl.atomic_add(grad_weight_ptr + weight_offsets, grad, mask=mask)


In [ ]:
def embedding_backward(
    grad_out: torch.Tensor,
    vocab_size: int,
    ids: torch.Tensor,
    BLOCK_M: int = 128,
    BLOCK_N: int = 128,
    check_indices: bool = False,
) -> torch.Tensor:
    """Method 02 backward: dW[I[m]] += G[m], accumulated with atomics."""
    grad_out = grad_out.contiguous()
    ids = ids.contiguous()

    if check_indices and ids.numel() > 0:
        assert ids.min() >= 0, "negative token id"
        assert ids.max() < vocab_size, "token id out of range"

    out_dtype = grad_out.dtype
    if out_dtype == torch.bfloat16:
        # bf16 atomics are slow/unsupported; accumulate in fp32, cast back after.
        grad_out = grad_out.float()

    ids_i32 = ids.to(torch.int32)
    grad_weight = torch.zeros(
        vocab_size, grad_out.shape[-1], device=grad_out.device, dtype=grad_out.dtype
    )
    n_tokens = ids.numel()
    if n_tokens == 0:
        return grad_weight.to(out_dtype)

    grid = (
        triton.cdiv(n_tokens, BLOCK_M),
        triton.cdiv(grad_out.shape[-1], BLOCK_N),
    )
    embedding_backward_kernel[grid](
        grad_out,
        grad_weight,
        ids_i32,
        n_tokens,
        grad_out.shape[-1],
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        DTYPE_FLAG=get_dtype_flag(grad_out.dtype),
    )
    return grad_weight.to(out_dtype)


In [ ]:
def _assert_close(actual: torch.Tensor, expected: torch.Tensor) -> None:
    """Tolerance depends on the dtype (fp16/bf16 are looser)."""
    if actual.dtype in (torch.float16, torch.bfloat16):
        torch.testing.assert_close(actual.float(), expected.float(), atol=2e-2, rtol=2e-2)
    else:
        torch.testing.assert_close(actual, expected, atol=1e-5, rtol=1e-5)


def test_correctness() -> None:
    """GPU: forward matches table[ids] for fp32/fp16/bf16 and odd sizes."""
    require_gpu()
    torch.manual_seed(0)
    for dtype in (torch.float32, torch.float16, torch.bfloat16):
        table = torch.randn(37, 19, device=DEVICE, dtype=dtype)
        ids = torch.randint(0, 37, (5, 7), device=DEVICE)
        out = embedding_forward(table, ids, BLOCK_M=4, BLOCK_N=8)
        _assert_close(out, table[ids])
        print(f"forward {dtype}: OK")

    # Non-contiguous table (a transposed view) must still work.
    big = torch.randn(19, 41, device=DEVICE)
    table_nc = big[:, :37].t()            # shape (37, 19), stride is not (19, 1)
    ids = torch.randint(0, 37, (3, 3), device=DEVICE)
    _assert_close(embedding_forward(table_nc, ids), table_nc[ids])
    print("forward non-contiguous table: OK")


def test_backward() -> None:
    """GPU: dW equals the scatter-add reference, including duplicate ids."""
    require_gpu()
    torch.manual_seed(0)
    for dtype in (torch.float32, torch.float16, torch.bfloat16):
        table = torch.randn(23, 11, device=DEVICE, dtype=dtype)
        ids = torch.randint(0, 23, (4, 6), device=DEVICE)
        grad = torch.randn(4, 6, 11, device=DEVICE, dtype=dtype)

        dW = embedding_backward(grad, table.shape[0], ids)
        ref = torch.zeros_like(table)
        ref.index_add_(0, ids.reshape(-1), grad.reshape(-1, 11))
        _assert_close(dW, ref)
        print(f"backward {dtype}: OK")

    # Many tokens share ids 0, 1 -> the atomic add must sum all of them.
    ids = torch.tensor([[0, 0, 0, 1, 1]], device=DEVICE)
    grad = torch.ones(1, 5, 7, device=DEVICE)
    dW = embedding_backward(grad, 3, ids)
    assert torch.allclose(dW[0], torch.full((7,), 3.0, device=DEVICE))
    assert torch.allclose(dW[1], torch.full((7,), 2.0, device=DEVICE))
    assert torch.allclose(dW[2], torch.zeros(7, device=DEVICE))
    print("backward duplicate-id collision: OK")


## Part IV — Method 03: fused kernel-based (lookup + position add)

The notes fuse two operations from page 1:

$$
H=X+\mathcal P=W[I]+\mathcal P.
$$

One kernel reads the table row, reads the position row, adds them in registers,
and writes a single output. This removes the intermediate $X$ tensor and one
full $M\times D$ write/read pair.

The backward stays a scatter-add for the table, and the position gradient is
simply $d\mathcal P=dH$ (same-shape case). For bf16, the wrapper promotes the
gradient to fp32 before `tl.atomic_add` and casts the result back afterward,
because bf16 atomics are slow or unsupported on many devices (the notes make
the same choice).


In [ ]:
@triton.jit
def fused_embedding_position_kernel(
    table_ptr,          # W: [V, D]
    ids_ptr,            # I: [M] int32
    pos_ptr,            # P: [M, D]
    out_ptr,            # H = W[I] + P: [M, D]
    n_tokens,
    embed_dim,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    if DTYPE_FLAG == 0:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.float32))
        pos_ptr = tl.cast(pos_ptr, tl.pointer_type(tl.float32))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.float32))
    elif DTYPE_FLAG == 1:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.float16))
        pos_ptr = tl.cast(pos_ptr, tl.pointer_type(tl.float16))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.float16))
    else:
        table_ptr = tl.cast(table_ptr, tl.pointer_type(tl.bfloat16))
        pos_ptr = tl.cast(pos_ptr, tl.pointer_type(tl.bfloat16))
        out_ptr = tl.cast(out_ptr, tl.pointer_type(tl.bfloat16))
    ids_ptr = tl.cast(ids_ptr, tl.pointer_type(tl.int32))

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < n_tokens
    col_mask = cols < embed_dim
    mask = row_mask[:, None] & col_mask[None, :]

    ids = tl.load(ids_ptr + rows, mask=row_mask, other=0)
    table_offsets = ids[:, None].to(tl.int64) * embed_dim + cols[None, :]
    values = tl.load(table_ptr + table_offsets, mask=mask, other=0.0)

    flat_offsets = rows[:, None].to(tl.int64) * embed_dim + cols[None, :]
    position = tl.load(pos_ptr + flat_offsets, mask=mask, other=0.0)

    # Add in fp32, then cast once: one store, and no intermediate X in HBM.
    total = values.to(tl.float32) + position.to(tl.float32)
    tl.store(out_ptr + flat_offsets, total.to(out_ptr.dtype.element_ty), mask=mask)


In [ ]:
def fused_embedding_forward(
    table: torch.Tensor,
    ids: torch.Tensor,
    position: torch.Tensor,
    BLOCK_M: int = 128,
    BLOCK_N: int = 128,
) -> torch.Tensor:
    """Method 03: H = W[I] + position in one kernel (same-shape position)."""
    assert table.ndim == 2, "table must be [V, D]"
    table = table.contiguous()
    ids = ids.contiguous()
    position = position.contiguous()
    assert position.shape == (*ids.shape, table.shape[1]), (
        f"position must have shape {(*ids.shape, table.shape[1])}, got {tuple(position.shape)}"
    )
    assert position.dtype == table.dtype, "position and table dtypes must match"

    out = torch.empty_like(position)
    n_tokens = ids.numel()
    if n_tokens == 0:
        return out

    grid = (
        triton.cdiv(n_tokens, BLOCK_M),
        triton.cdiv(table.shape[1], BLOCK_N),
    )
    fused_embedding_position_kernel[grid](
        table,
        ids.to(torch.int32),
        position,
        out,
        n_tokens,
        table.shape[1],
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        DTYPE_FLAG=get_dtype_flag(table.dtype),
    )
    return out


def fused_embedding_backward(
    grad_out: torch.Tensor,
    vocab_size: int,
    ids: torch.Tensor,
    BLOCK_M: int = 128,
    BLOCK_N: int = 128,
):
    """Method 03 backward: dW is a scatter-add; dP = dH for a same-shape position."""
    d_weight = embedding_backward(grad_out, vocab_size, ids, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    d_position = grad_out.clone()   # dH/dP = 1
    return d_weight, d_position


In [ ]:
def test_fused_position() -> None:
    """GPU: fused H = W[I] + P matches PyTorch and its gradient is correct."""
    require_gpu()
    torch.manual_seed(0)
    for dtype in (torch.float32, torch.float16, torch.bfloat16):
        table = torch.randn(29, 13, device=DEVICE, dtype=dtype)
        ids = torch.randint(0, 29, (3, 5), device=DEVICE)
        position = torch.randn(3, 5, 13, device=DEVICE, dtype=dtype)

        out = fused_embedding_forward(table, ids, position, BLOCK_M=4, BLOCK_N=8)
        _assert_close(out, table[ids] + position)

        grad = torch.randn_like(out)
        d_weight, d_position = fused_embedding_backward(grad, table.shape[0], ids)
        ref_dw = torch.zeros_like(table)
        ref_dw.index_add_(0, ids.reshape(-1), grad.reshape(-1, 13))
        _assert_close(d_weight, ref_dw)
        _assert_close(d_position, grad)
        print(f"fused forward/backward {dtype}: OK")


In [ ]:
def embedding_bytes(vocab: int, dim: int, tokens: int, itemsize: int):
    """Analytic traffic for one forward + one backward (bytes)."""
    forward = 2 * tokens * dim * itemsize          # read rows + write output
    backward = (vocab * dim + tokens * dim) * itemsize  # touch table + read grad
    return forward, backward


V, D, M = 50_000, 1_024, 2_048
fwd, bwd = embedding_bytes(V, D, M, itemsize=2)   # fp16
print(f"forward traffic : {fwd / 1e6:8.2f} MB")
print(f"backward traffic: {bwd / 1e6:8.2f} MB")
print(f"backward / forward ratio: {bwd / fwd:,.1f}x")
print(f"backward table alone    : {V * D * 2 / 1e6:8.2f} MB")


## Pitfalls, checks, and exercises

**Pitfalls**

- Forgetting the row mask when loading ids → out-of-bounds reads on the last tile.
- Forgetting the column mask when $D$ is not a multiple of `BLOCK_N`.
- Using `tl.store` in the backward → duplicate ids race and lose gradients.
- Accumulating bf16 atomics directly → slow and/or unsupported; accumulate fp32.
- Assuming `dW` is sparse: `torch.zeros_like(table)` still writes $VD$ bytes.
- Negative ids: PyTorch wraps them, but `nn.Embedding` rejects them; check or document the contract.
- Broadcasting a shared position table without summing the gradient over the broadcast axis.

**Self-check**

1. For $V=5,D=3,M=4$ and $I=[2,0,1,2]$, write $P$, verify $X=PW$, and compute $dW$ for $G=\mathbf 1$.
2. Show that $dW=P^\top G$ and that row $j$ of $dW$ is the sum of the $G_m$ with $I_m=j$.
3. Why is `tl.atomic_add` needed? Give two token ids that collide.
4. Derive the forward/backward byte counts for $V=50{,}000$, $D=1024$, $M=2048$, fp16.
5. Change `BLOCK_M`/`BLOCK_N` and confirm the masks still cover every element.
6. Modify the fused kernel so a position table of shape `(1, D)` broadcasts over tokens; what must the backward do?


In [ ]:
print("Running CPU tests ...")
test_pseudo_matches_torch()
test_worked_example()

print("\nRunning GPU tests (skipped without CUDA/ROCm) ...")
for fn in (test_correctness, test_backward, test_fused_position):
    try:
        fn()
    except pytest.skip.Exception as exc:
        print(f"SKIP {fn.__name__}: {exc}")
